In [1]:
!pip install tiktoken
!pip install pydantic
!pip install openai
# !pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 14.0 MB/s eta 0:00:00


In [52]:
class Participant(BaseModel):
  """
  Represents one of the participants of the audio recording
  """
  speaker_id: str = Field(...,description="SPEAKER number of the participant during the audio")
  name: str = Field(..., description="Nickname of the participant. Probably mentioned by other speaker or himself")
  role: str = Field(..., description="Role of the participant during the scenario. Mentioned by himself during the presentation")

class Presentation(BaseModel):
  """
  Represents when a participant presents their role
  """
  speaker: Participant = Field(...,description="Participant")
  start_timestamp: Optional[float] = Field(..., description="Start time when speaker starts presenting")
  stop_timestamp: Optional[float] = Field(..., description="Stop time when participant finishes their sentence")


class Order(BaseModel):
  """
  Represents what a participant orders
  """
  leader: Participant = Field(...,description="Leader participant that orders the action")
  follower: Optional[Participant] = Field(...,description="Participant that does the action. It could be on his/her own or ordered by other participant")
  action: str = Field(..., description="Action ordered by leader")
  start_timestamp: Optional[float] = Field(..., description="Start time when leader ordered the action")
  stop_timestamp: Optional[float] = Field(..., description="Stop time when participant mentioned something about his/her action")

class Action(BaseModel):
  """
  Represents what a participant is doing
  """
  speaker: Participant = Field(...,description="Participant does the action")
  action: str = Field(..., description="Action taken by the participant")
  start_timestamp: Optional[float] = Field(..., description="Start time when participant is doing something")
  stop_timestamp: Optional[float] = Field(..., description="Stop time when participant finishes the action")

class Skill(BaseModel):
  """
  Represents a non-technical skill of a participant in a certain moment
  """
  speaker: Participant = Field(..., description= "Participant with a principle of skill")
  main_skill: str = Field(..., description = "Main Non-technical skill or principle that the participant is showing")
  sentence: str = Field(..., description = "Sentence that reflects the non-technical skill")
  skill_description: str = Field(..., description = "Additional information about Non-technical skill")
  start_timestamp: Optional[float] = Field(..., description="Start time when participant is showing a skill")
  stop_timestamp: Optional[float] = Field(..., description="Stop time when participant finishes the sentence that shows a skill")

class Orders(BaseModel):
    #presentationsTimeline: list[Presentation]
    ordersTimeline: list[Order]

class Scenario(BaseModel):
    actionsTimeline: list[Action]

class CRMSkills(BaseModel):
    skillsTimeline: list[Skill]

In [53]:
import os
from pydantic import BaseModel, Field
from openai import OpenAI
from typing import Optional, List
import json


In [276]:
# with open("results_whisperx_speakers.txt") as f:
#     transcription = json.load(f)
req_iteration = "5"
scenario = "crmscripted"
file = open("transcription_manual_"+scenario+".txt", "r")
transcriptionTxt = "Contenido: " + file.read()  # ✅ Esto convierte el contenido en una cadena
file.close()

In [ ]:

# https://platform.openai.com/api-keys
# Billing: https://platform.openai.com/settings/organization/billing/overview
OPENAI_API_KEY = 'OPENAI_API_KEY'
os.environ['OPEN_API_KEY'] = OPENAI_API_KEY
client = OpenAI(api_key=OPENAI_API_KEY)

In [279]:
# https://community.openai.com/t/api-access-using-free-tier/710656  => OpenAI does not allow to use free API
# RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

# https://platform.openai.com/docs/guides/structured-outputs?api-mode=chat
# Structured Outputs is available in our latest large language models, starting with GPT-4o:
# gpt-4.5-preview-2025-02-27 and later
# o3-mini-2025-1-31 and later
# o1-2024-12-17 and later
# gpt-4o-mini-2024-07-18 and later
# gpt-4o-2024-08-06 and later


def extractOrders(content, sys_mess, model="gpt-4o-2024-08-06", temperature=0):
  response = client.beta.chat.completions.parse(
      model=model,
      temperature=temperature,
      messages=[
          {"role": "system", "content": sys_mess},
          {"role": "user", "content": content}
      ],
      response_format=Orders
  )
  return json.loads(response.choices[0].message.content)

def extractScenario(content, sys_mess, model="gpt-4o-2024-08-06", temperature=0):
  response = client.beta.chat.completions.parse(
      model=model,
      temperature=temperature,
      messages=[
          {"role": "system", "content": sys_mess},
          {"role": "user", "content": content}
      ],
      response_format=Scenario
  )
  return json.loads(response.choices[0].message.content)

def extractSkills(content, sys_mess, model="gpt-4o-2024-08-06", temperature=0):
  response = client.beta.chat.completions.parse(
      model=model,
      temperature=temperature,
      messages=[
          {"role": "system", "content": sys_mess},
          {"role": "user", "content": content}
      ],
      response_format=CRMSkills
  )
  return json.loads(response.choices[0].message.content)


In [280]:
system_message = """
You are an expert in extracting structured information from audio transcriptions for downstream processing. Extract the following elements:
+ Participants:
  - speaker_id: The unique ID (e.g., SPEAKER_01).
  - name: If mentioned by themselves or others.
  - role: If introduced or inferred from conversation context.
+ Orders: Analyze all the utterances to extract orders. Orders are defined by imperative sentences (implied "you" subject + infinitive verb) or directly mentioning someone's name. May be phrased as questions or directives about actions, resources, environment, historical processes, current situation, etc (e.g., "Can you check the report?", "What are his vitals?", "Provide paperwork", "Prepare drugs").
  Include:
  - leader: Who gave the order.
  - follower: Who the order was directed to (can be inferred from replies or context, but sometimes there is no answer).
  - action: A brief description of the order.
  - start_time: Timestamp in seconds when the order begins.
  - end_time: Timestamp when the order ends.
Inference Rules: If names/roles are not explicitly stated, try to infer them from context. For orders, cross-reference the following utterances for replies to infer recipients. Set to "Unknown" if not reasonably inferable; do not hallucinate.

Present the extracted information in a clear, structured format. Be concise, focusing on:
- Identifying all the participants
- Identifying the orders between participants
- Identify the timestamps when everything happened
"""

In [281]:

import time

start_time = time.time()
data = extractOrders(transcriptionTxt, system_message)
end_time = time.time()
processing_duration = end_time - start_time
print(f"Processed in {processing_duration:.2f} seconds")


Processed in 7.61 seconds


In [282]:
print(len(data["ordersTimeline"]))

9


In [283]:

with open("orders_"+scenario+req_iteration+".json", 'w') as f:
    json.dump(data, f)

In [287]:
system_message = """
You are an expert in analyzing transcriptions to extract non-technical skills for downstream processing. Extract the following elements:
+ Participants:
  - speaker_id: Unique identifier (e.g., SPEAKER_01).
  - name: If mentioned by themselves or another speaker.
  - role: If introduced explicitly or can be inferred from context.
+ Non-Technical Skills: Detect non-technical skills exhibited by participants in each utterance. Multiple skills may apply to a single utterance. Skill Categories:
  1. Team Management:
    - Leadership & followership
    - Role assignment
    - Giving orders or commands
    - Task distribution
    - Requesting help
    - Clear communication to others
  2. Resource Allocation:
    - Accessing or using equipment, tools, machines
    - Running procedures
    - Handling or interacting with objects
  3. Environmental Awareness:
    - Planning or delivering next steps
    - Reviewing past or current actions
    - Mentioning surrounding conditions
  4. Dynamic Decision-Making:
    - Changing a decision
    - Discussing or choosing next steps
    - Using available information (e.g., history, monitor, reports)
    - Justifying decisions based on prior events
  For each skill occurrence, include:
  - speaker: Who exhibited the skill
  - skill_type: One of ["Team Management", "Resource Allocation", "Environmental Awareness", "Dynamic Decision-Making"]
  - description: Subcategory of the skill_type
  - start_time: Timestamp when the skill occurs (in seconds)
  - end_time: Timestamp when it ends (in seconds)
Inference Rules: If names or roles are not directly mentioned, infer from context when possible. If a skill is not explicitly stated, analyze both the utterance and responses around it. Use the skill classification list to guide inference. Skip if uncertain and no inference is reasonably possible.

Present the extracted information in a clear, structured format. Be concise, focusing on:
- Identifying all the participants
- Identifying the non-technical skills present in each speaker words
- Identify the timestamps when everything happened
"""

In [288]:

import time

start_time = time.time()
data = extractSkills(transcriptionTxt, system_message)
end_time = time.time()
processing_duration = end_time - start_time
print(f"Processed in {processing_duration:.2f} seconds")


Processed in 22.49 seconds


In [289]:
print(len(data["skillsTimeline"]))

22


In [290]:

with open("skills_"+scenario+req_iteration+".json", 'w') as f:
    json.dump(data, f)